In [1]:
# ============================================================
# CELL 1 — LIGHTGBM TRAINING SETUP
# ============================================================

import os
import json
import numpy as np
import pandas as pd

print("=" * 60)
print("MODEL 2 — LIGHTGBM TRAINING")
print("=" * 60)

# ------------------------------------------------------------
# Base directory
# ------------------------------------------------------------

BASE_DIR = os.getcwd()

# ------------------------------------------------------------
# Preprocessed data
# ------------------------------------------------------------

PREPROCESSED_DIR = os.path.join(
    BASE_DIR,
    "model2_preprocessed_lightgbm"
)

EMBEDDINGS_PATH = os.path.join(
    PREPROCESSED_DIR,
    "embeddings.npy"
)

DATASET_PATH = os.path.join(
    PREPROCESSED_DIR,
    "dataset.csv"
)

LABEL_MAPPING_PATH = os.path.join(
    PREPROCESSED_DIR,
    "label_mapping.json"
)

CLASS_WEIGHTS_PATH = os.path.join(
    PREPROCESSED_DIR,
    "class_weights.json"
)

# ------------------------------------------------------------
# Model output
# ------------------------------------------------------------

OUTPUT_DIR = os.path.join(
    BASE_DIR,
    "outputs",
    "model2_lightgbm",
    "final"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

MODEL_PATH = os.path.join(
    OUTPUT_DIR,
    "model.txt"
)

# ------------------------------------------------------------
# Display paths
# ------------------------------------------------------------

print("\nBase directory:")
print(BASE_DIR)

print("\nPreprocessed data:")
print(PREPROCESSED_DIR)

print("\nModel output:")
print(OUTPUT_DIR)

print("\nModel will be saved to:")
print(MODEL_PATH)

MODEL 2 — LIGHTGBM TRAINING

Base directory:
c:\Users\astha\Desktop\sih2026\Modular-medical-intelligence-and-Dialogue-system\model_2(Medical Dialogue Manager)

Preprocessed data:
c:\Users\astha\Desktop\sih2026\Modular-medical-intelligence-and-Dialogue-system\model_2(Medical Dialogue Manager)\model2_preprocessed_lightgbm

Model output:
c:\Users\astha\Desktop\sih2026\Modular-medical-intelligence-and-Dialogue-system\model_2(Medical Dialogue Manager)\outputs\model2_lightgbm\final

Model will be saved to:
c:\Users\astha\Desktop\sih2026\Modular-medical-intelligence-and-Dialogue-system\model_2(Medical Dialogue Manager)\outputs\model2_lightgbm\final\model.txt


In [3]:
# ============================================================
# CELL 2 — LOAD PREPROCESSED DATA
# ============================================================

# ------------------------------------------------------------
# Load embeddings
# ------------------------------------------------------------

X = np.load(
    EMBEDDINGS_PATH
)

# ------------------------------------------------------------
# Load dataset
# ------------------------------------------------------------

df = pd.read_csv(
    DATASET_PATH
)

# ------------------------------------------------------------
# Load label mapping
# ------------------------------------------------------------

with open(
    LABEL_MAPPING_PATH,
    "r",
    encoding="utf-8"
) as f:
    label_mapping = json.load(f)

label2id = label_mapping["label2id"]
id2label = label_mapping["id2label"]

# ------------------------------------------------------------
# Load class weights
# ------------------------------------------------------------

with open(
    CLASS_WEIGHTS_PATH,
    "r",
    encoding="utf-8"
) as f:
    class_weights_named = json.load(f)

# ------------------------------------------------------------
# Create numeric class weights
# ------------------------------------------------------------

class_weights = {
    int(label2id[label]): float(weight)
    for label, weight in class_weights_named.items()
}

# ------------------------------------------------------------
# Prepare target labels
# ------------------------------------------------------------

y = df["label_id"].values

# ------------------------------------------------------------
# Display information
# ------------------------------------------------------------

print("=" * 60)
print("PREPROCESSED DATA LOADED")
print("=" * 60)

print("\nEmbeddings shape:")
print(X.shape)

print("\nNumber of records:")
print(len(df))

print("\nNumber of classes:")
print(len(label2id))

print("\nTarget shape:")
print(y.shape)

print("\nNumber of class weights:")
print(len(class_weights))

print("\nFirst 5 labels:")
for class_id in sorted(id2label.keys(), key=int)[:5]:
    print(f"{class_id} -> {id2label[class_id]}")

print("\nData loading successful.")

PREPROCESSED DATA LOADED

Embeddings shape:
(100, 768)

Number of records:
100

Number of classes:
36

Target shape:
(100,)

Number of class weights:
36

First 5 labels:
0 -> RED_FLAG
1 -> associated_symptoms
2 -> breath_onset
3 -> breath_progression
4 -> breath_severity

Data loading successful.


In [4]:
# ============================================================
# CELL 3 — CREATE SAMPLE WEIGHTS
# ============================================================

# ------------------------------------------------------------
# Assign each training example the weight of its class
# ------------------------------------------------------------

sample_weights = np.array([
    class_weights[int(label_id)]
    for label_id in y
])

# ------------------------------------------------------------
# Verify
# ------------------------------------------------------------

print("=" * 60)
print("SAMPLE WEIGHTS CREATED")
print("=" * 60)

print("\nNumber of samples:")
print(len(sample_weights))

print("\nSample weights shape:")
print(sample_weights.shape)

print("\nMinimum weight:")
print(sample_weights.min())

print("\nMaximum weight:")
print(sample_weights.max())

print("\nFirst 10 examples:")
for i in range(10):
    label_id = int(y[i])
    label = id2label[str(label_id)]
    weight = sample_weights[i]

    print(
        f"{i+1:2d}. "
        f"{label:30s} | "
        f"Weight: {weight:.4f}"
    )

# ------------------------------------------------------------
# Sanity check
# ------------------------------------------------------------

assert len(sample_weights) == len(X)
assert len(sample_weights) == len(y)

print("\nSample-weight verification: PASSED")

SAMPLE WEIGHTS CREATED

Number of samples:
100

Sample weights shape:
(100,)

Minimum weight:
0.1111111111111111

Maximum weight:
2.7777777777777777

First 10 examples:
 1. pain_location                  | Weight: 0.3968
 2. pain_quality                   | Weight: 0.5556
 3. pain_severity                  | Weight: 0.4630
 4. associated_symptoms            | Weight: 0.1111
 5. pain_quality                   | Weight: 0.5556
 6. pain_progression               | Weight: 1.3889
 7. associated_symptoms            | Weight: 0.1111
 8. pain_triggers                  | Weight: 2.7778
 9. pain_location                  | Weight: 0.3968
10. pain_quality                   | Weight: 0.5556

Sample-weight verification: PASSED


In [7]:
# ============================================================
# CELL 4 — CREATE LIGHTGBM CLASSIFIER
# ============================================================

import lightgbm as lgb

# ------------------------------------------------------------
# Number of classes
# ------------------------------------------------------------

NUM_CLASSES = len(label2id)

# ------------------------------------------------------------
# Create multiclass LightGBM classifier
# ------------------------------------------------------------

model = lgb.LGBMClassifier(
    objective="multiclass",
    num_class=NUM_CLASSES,

    # Tree configuration
    n_estimators=200,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,

    # Feature / split configuration
    min_child_samples=5,

    # Reproducibility
    random_state=42,

    # Prevent unnecessary console output
    verbosity=-1
)

print("=" * 60)
print("LIGHTGBM CLASSIFIER CREATED")
print("=" * 60)

print("\nObjective:")
print("multiclass")

print("\nNumber of classes:")
print(NUM_CLASSES)

print("\nNumber of estimators:")
print(model.n_estimators)

print("\nLearning rate:")
print(model.learning_rate)

print("\nNumber of leaves:")
print(model.num_leaves)

print("\nLightGBM version:")
print(lgb.__version__)

LIGHTGBM CLASSIFIER CREATED

Objective:
multiclass

Number of classes:
36

Number of estimators:
200

Learning rate:
0.05

Number of leaves:
31

LightGBM version:
4.7.0


In [8]:
# ============================================================
# CELL 5 — TRAIN LIGHTGBM
# ============================================================

print("=" * 60)
print("STARTING LIGHTGBM TRAINING")
print("=" * 60)

print("\nTraining samples:", X.shape[0])
print("Input features:", X.shape[1])
print("Number of classes:", NUM_CLASSES)

print("\nClass weighting: ENABLED")
print("Train/validation split: NOT USED")
print("\nTraining...\n")

# ------------------------------------------------------------
# Train the model
# ------------------------------------------------------------

model.fit(
    X,
    y,
    sample_weight=sample_weights
)

print("\n" + "=" * 60)
print("LIGHTGBM TRAINING COMPLETED")
print("=" * 60)

STARTING LIGHTGBM TRAINING

Training samples: 100
Input features: 768
Number of classes: 36

Class weighting: ENABLED
Train/validation split: NOT USED

Training...


LIGHTGBM TRAINING COMPLETED


In [9]:
# ============================================================
# CELL 6 — SAVE TRAINED LIGHTGBM MODEL
# ============================================================

model.booster_.save_model(
    MODEL_PATH
)

print("=" * 60)
print("LIGHTGBM MODEL SAVED")
print("=" * 60)

print("\nModel path:")
print(MODEL_PATH)

print("\nModel file exists:")
print(os.path.exists(MODEL_PATH))

if os.path.exists(MODEL_PATH):
    model_size_mb = os.path.getsize(MODEL_PATH) / (1024 * 1024)
    print(f"Model size: {model_size_mb:.2f} MB")

LIGHTGBM MODEL SAVED

Model path:
c:\Users\astha\Desktop\sih2026\Modular-medical-intelligence-and-Dialogue-system\model_2(Medical Dialogue Manager)\outputs\model2_lightgbm\final\model.txt

Model file exists:
True
Model size: 6.12 MB
